# 1.データ読み込み

In [214]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [215]:
import pandas as pd
df = pd.read_excel('/content/drive/MyDrive/NTCIR-19/生成データ_頭痛_サンプル.xlsx')
df

,id,会話,痛み,しびれ,振る舞い,トリアージ
0,1,Dispatcher: 救急ですか、火事ですか。\nCaller: 救急です。夫の様子がおか...,0,0,0,1
1,2,Dispatcher: 救急ですか、火事ですか。\nCaller: 救急です。職場の同僚がひ...,0,0,0,1
2,3,Dispatcher: 救急ですか、火事ですか。\nCaller: 救急です。入所者さんが急...,0,0,0,1
3,4,Dispatcher: 救急ですか、火事ですか。\nCaller: 救急です。近所の方が頭が...,2,2,2,0
4,5,Dispatcher: 救急ですか、火事ですか。\nCaller: 救急です。学生がひどい頭...,2,2,2,0
5,6,Dispatcher: 救急ですか、火事ですか。\nCaller: 救急です。うちの従業員が...,1,1,1,2
6,7,Dispatcher: 救急ですか、火事ですか。\nCaller: 救急です。友人がひどい頭...,1,1,1,2


# 2.会話をラリーごとに分割し、質問と回答のペアを作成

In [216]:
import re

def parse_conversation(conversation_text):
    # 正規表現パターンを1行で正しく定義 (文字通りの改行文字を避けるため)
    dispatcher_turns = re.findall(r'Dispatcher:([^\n]*)', conversation_text)
    caller_turns = re.findall(r'Caller:([^\n]*)', conversation_text)

    qa_pairs = []
    min_len = min(len(dispatcher_turns), len(caller_turns))

    for i in range(min_len):
        question = dispatcher_turns[i].strip()
        answer = caller_turns[i].strip()
        if question and answer:
            qa_pairs.append({'質問': question, '回答': answer})

    return qa_pairs

# 1. 各会話ログに対して関数を適用し、結果を新しいカラムに格納
df['qa_pairs'] = df['会話'].apply(parse_conversation)

# 2. qa_pairsカラムを展開して新しいDataFrameを作成
df_expanded_combined = df.explode('qa_pairs')

# 3. 展開されたqa_pairsからquestionとanswerを抽出して新しいカラムとして追加
# ここでキーを'質問'と'回答'に修正し、新しいカラム名もこれに合わせます。
df_expanded_combined['質問'] = df_expanded_combined['qa_pairs'].apply(lambda x: x['質問'] if isinstance(x, dict) else None)
df_expanded_combined['回答'] = df_expanded_combined['qa_pairs'].apply(lambda x: x['回答'] if isinstance(x, dict) else None)

# 4. 元の'会話'と'qa_pairs'カラムは不要なので削除
df_pairs = df_expanded_combined.drop(columns=['会話', 'qa_pairs'])
df_pairs['ペア'] = df_pairs['質問'] + " " + df_pairs['回答']

# 新しい「ペア番号」カラムを追加 (idごとにシーケンス番号を振る)
df_pairs['ペア番号'] = df_pairs.groupby('id').cumcount()

# 5. カラムの順序を再編成
ordered_columns = ['id', 'ペア番号', 'ペア', '痛み', 'しびれ', '振る舞い', 'トリアージ', '質問', '回答']
df_pairs = df_pairs[ordered_columns]

# DataFrameのインデックスをリセットして一意にする
df_pairs = df_pairs.reset_index(drop=True)

# 結果の表示
display(df_pairs)

,id,ペア番号,ペア,痛み,しびれ,振る舞い,トリアージ,質問,回答
0,1,0,救急ですか、火事ですか。 救急です。夫の様子がおかしいです。,0,0,0,1,救急ですか、火事ですか。,救急です。夫の様子がおかしいです。
1,1,1,どうしましたか。 さっきテレビを見ていたら、急に頭を押さえて、ものすごく痛いって言い出しまし...,0,0,0,1,どうしましたか。,さっきテレビを見ていたら、急に頭を押さえて、ものすごく痛いって言い出しました。頭痛です。
2,1,2,その頭の痛みは、急に強く出たんですね。 はい、突然です。今までにないくらい痛いと言っています。,0,0,0,1,その頭の痛みは、急に強く出たんですね。,はい、突然です。今までにないくらい痛いと言っています。
3,1,3,手足のしびれや、片側が動かしにくい感じはありますか。 あります。右手がうまく動かないみたいで...,0,0,0,1,手足のしびれや、片側が動かしにくい感じはありますか。,あります。右手がうまく動かないみたいで、しびれるとも言っています。
4,1,4,いつもと違って、受け答えがおかしいとか、様子が変だということはありますか。 はい、あります。...,0,0,0,1,いつもと違って、受け答えがおかしいとか、様子が変だということはありますか。,はい、あります。呼びかけても返事が遅いですし、言葉も少し変で、ぼんやりしています。
5,2,0,救急ですか、火事ですか。 救急です。職場の同僚がひどい頭痛で倒れ込みました。,0,0,0,1,救急ですか、火事ですか。,救急です。職場の同僚がひどい頭痛で倒れ込みました。
6,2,1,どうしましたか。 休憩中に急に頭が割れるみたいに痛いと言って、その場にしゃがみ込んでしまいました。,0,0,0,1,どうしましたか。,休憩中に急に頭が割れるみたいに痛いと言って、その場にしゃがみ込んでしまいました。
7,2,2,痛みはじわじわではなく、急に始まった感じですか。 はい、急でした。普通に話していたのに、突然です。,0,0,0,1,痛みはじわじわではなく、急に始まった感じですか。,はい、急でした。普通に話していたのに、突然です。
8,2,3,しびれたり、顔や腕が動かしにくそうな様子はありますか。 左腕がしびれると言っています。コップ...,0,0,0,1,しびれたり、顔や腕が動かしにくそうな様子はありますか。,左腕がしびれると言っています。コップも落としてしまいました。
9,2,4,普段と違う行動や、会話がかみ合わない感じはありますか。 あります。名前を呼んでも反応が鈍いし...,0,0,0,1,普段と違う行動や、会話がかみ合わない感じはありますか。,あります。名前を呼んでも反応が鈍いし、何を聞いても同じことを繰り返しています。


# 3.ペアとひし形のベクトル化

## 3.1.Sentence BERTの準備

文章のベクトル化にはSentence BERTを使うらしい

In [46]:
# !pip install transformers sentencepiece fugashi ipadic

In [47]:
from transformers import MLukeTokenizer, LukeModel
import torch


class SentenceLukeJapanese:
    def __init__(self, model_name_or_path, device=None):
        self.tokenizer = MLukeTokenizer.from_pretrained(model_name_or_path)
        self.model = LukeModel.from_pretrained(model_name_or_path)
        self.model.eval()

        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = torch.device(device)
        self.model.to(device)

    def _mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output[0] #First element of model_output contains all token embeddings
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    @torch.no_grad()
    def encode(self, sentences, batch_size=8):
        all_embeddings = []
        iterator = range(0, len(sentences), batch_size)
        for batch_idx in iterator:
            batch = sentences[batch_idx:batch_idx + batch_size]

            # Use the tokenizer's __call__ method for encoding
            encoded_input = self.tokenizer(batch, padding="longest",
                                           truncation=True, return_tensors="pt").to(self.device)
            model_output = self.model(**encoded_input)
            sentence_embeddings = self._mean_pooling(model_output, encoded_input["attention_mask"]).to('cpu')

            all_embeddings.extend(sentence_embeddings)

        return torch.stack(all_embeddings)

## 3.2.ペアのベクトル化

In [217]:
MODEL_NAME = "sonoisa/sentence-luke-japanese-base-lite"
model_luke = SentenceLukeJapanese(MODEL_NAME)

# tqdmをインポートして進行状況バーを表示
from tqdm.notebook import tqdm
tqdm.pandas()

# 'ペア'カラムのテキストをベクトル化
# NaN値がある場合はNoneを返す
embeddings = df_pairs['ペア'].progress_apply(lambda x: model_luke.encode([x])[0].tolist() if pd.notna(x) else None)

# ベクトルを新しいカラムとしてDataFrameに追加
df_pairs['ペアのベクトル'] = embeddings

# 結果の表示（関連カラムのみ）
display(df_pairs[['id','ペア番号', 'ペア', 'ペアのベクトル']])

Loading weights:   0%|          | 0/277 [00:00<?, ?it/s]

  0%|          | 0/35 [00:00<?, ?it/s]

,id,ペア番号,ペア,ペアのベクトル
0,1,0,救急ですか、火事ですか。 救急です。夫の様子がおかしいです。,"[-0.07553009688854218, 0.6543946266174316, -0...."
1,1,1,どうしましたか。 さっきテレビを見ていたら、急に頭を押さえて、ものすごく痛いって言い出しまし...,"[-0.5534268617630005, 0.2958027720451355, 0.28..."
2,1,2,その頭の痛みは、急に強く出たんですね。 はい、突然です。今までにないくらい痛いと言っています。,"[-0.4428359568119049, 0.2741571366786957, 0.06..."
3,1,3,手足のしびれや、片側が動かしにくい感じはありますか。 あります。右手がうまく動かないみたいで...,"[-0.2704547345638275, -0.16498802602291107, 0...."
4,1,4,いつもと違って、受け答えがおかしいとか、様子が変だということはありますか。 はい、あります。...,"[-0.264884889125824, -0.10918011516332626, 0.2..."
5,2,0,救急ですか、火事ですか。 救急です。職場の同僚がひどい頭痛で倒れ込みました。,"[-0.484037846326828, 0.34421274065971375, 0.07..."
6,2,1,どうしましたか。 休憩中に急に頭が割れるみたいに痛いと言って、その場にしゃがみ込んでしまいました。,"[-0.1401555985212326, 0.48630091547966003, 0.0..."
7,2,2,痛みはじわじわではなく、急に始まった感じですか。 はい、急でした。普通に話していたのに、突然です。,"[0.2931731641292572, 0.1633119285106659, 0.148..."
8,2,3,しびれたり、顔や腕が動かしにくそうな様子はありますか。 左腕がしびれると言っています。コップ...,"[-0.054328955709934235, -0.4805036187171936, 0..."
9,2,4,普段と違う行動や、会話がかみ合わない感じはありますか。 あります。名前を呼んでも反応が鈍いし...,"[-0.3791346549987793, -0.07827914506196976, 0...."


##3.3.ひし形のベクトル化

### 3.3.1.ひし形の質問事項の準備

In [218]:
import pandas as pd

hisigata = pd.DataFrame({
    'ひし形_全通り_頭痛_1': ['激しい痛みが、起こりましたか？'],
    'ひし形_全通り_頭痛_2': ['しびれや麻痺がありますか？'],
    'ひし形_全通り_頭痛_3': ['何か、いつもと違う振る舞いがありますか？（発症から3時間以内 ）'],
    'ひし形_直接': [
        'あなたのトリアージレベルはどれに当てはまりますか？\n'
        'R1:心肺蘇生の必要性が強く疑われる病態\n'
        'R2:高度な医学的判断・処置の必要性が高く、その開始までの時間に急を要する病態\n'
        'R3:高度な医学的判断・処置の必要性は R2 より低いが、迅速な到着と搬送が必要な病態\n'
        'Y1:医学的判断の必要性は高いが、R2・3 ほどの迅速性は必要ない病態\n'
        'Y2:医学的判断の必要性は R1～Y1 ほど高くないが、医療機関への受診が必要な病態\n'
        'G:R、Yには該当しないが、診察が必要な病態'
    ]
})

# 頭痛以外の症状をするときには、ひし形_nみたいにかくちょうできるんやね？って今は思っとるよ。もっといいやり方あるかたいきにまた聞こう

### 3.3.2.ひし形のベクトル化

In [219]:
embeddings_hisigata = {}
for col_name in hisigata.columns:
    question_text = hisigata[col_name].iloc[0]
    # embedding_ひし形_1のような変数名で個別の埋め込みも作成する（既存コードとの互換性のため）
    globals()[f'embedding_{col_name}'] = model_luke.encode([question_text])[0].tolist()
    embeddings_hisigata[f'embedding_{col_name}'] = globals()[f'embedding_{col_name}']

# 印刷部分も動的に変更
for col_name in hisigata.columns:
    print(f"{col_name} の埋め込み:")
    print(embeddings_hisigata[f'embedding_{col_name}'])
    print()

ひし形_全通り_頭痛_1 の埋め込み:
[0.32222095131874084, -0.22027680277824402, 0.2368488758802414, -0.231918603181839, -0.40979060530662537, 0.2133362591266632, -0.1511187106370926, 0.11258430778980255, 0.15832829475402832, -0.5831359624862671, 7.77319073677063e-05, -0.21630465984344482, -0.32570868730545044, 0.3361362814903259, -0.006939095910638571, 0.21191339194774628, 0.022148044779896736, 0.037182729691267014, -0.13453112542629242, 0.19073733687400818, 0.43522095680236816, -0.1366681158542633, 0.18855871260166168, -0.011761264875531197, 0.07846628129482269, 0.12858735024929047, -0.9526805877685547, -0.31885358691215515, -0.3015802800655365, 0.07574518769979477, -0.09436048567295074, -0.28341805934906006, 0.2651134133338928, 0.3863716721534729, -0.08626748621463776, -0.007021838333457708, -0.3862331211566925, -0.003976141102612019, -0.1530579775571823, -0.2701128125190735, 0.11910592019557953, -0.281715452671051, -0.43905797600746155, -0.7646943926811218, -0.036851707845926285, -0.687809884548187

# 4.入力するペアの選定

## 4.1.ひし形と各ペアのcos類似度計算

In [220]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# df_pairsのペアのベクトルカラムをnumpy配列に変換
conversation_embeddings = np.array(df_pairs['ペアのベクトル'].tolist())

# 各ひし形と会話ペアのコサイン類似度を計算し、df_pairsに直接追加
for hisigata_name_key, embedding_list in embeddings_hisigata.items():
    # hisigata_name_keyは 'embedding_ひし形_1' のような文字列なので、 'ひし形_1' を抽出
    original_hisigata_name = hisigata_name_key.replace('embedding_', '')
    embedding_np = np.array(embedding_list).reshape(1, -1)
    df_pairs[f'cos_sim_{original_hisigata_name}'] = cosine_similarity(conversation_embeddings, embedding_np).flatten()

# 結果の表示（関連カラムのみ）
# 表示カラムも動的に生成
display_cols = ['id', 'ペア番号', 'ペア']
for col_name in hisigata.columns:
    display_cols.append(f'cos_sim_{col_name}')
display(df_pairs[display_cols])

,id,ペア番号,ペア,cos_sim_ひし形_全通り_頭痛_1,cos_sim_ひし形_全通り_頭痛_2,cos_sim_ひし形_全通り_頭痛_3,cos_sim_ひし形_直接
0,1,0,救急ですか、火事ですか。 救急です。夫の様子がおかしいです。,0.333049,0.114088,0.417469,0.579390
1,1,1,どうしましたか。 さっきテレビを見ていたら、急に頭を押さえて、ものすごく痛いって言い出しまし...,0.371689,0.142192,0.481872,0.500633
2,1,2,その頭の痛みは、急に強く出たんですね。 はい、突然です。今までにないくらい痛いと言っています。,0.343067,0.164993,0.551439,0.481516
3,1,3,手足のしびれや、片側が動かしにくい感じはありますか。 あります。右手がうまく動かないみたいで...,0.372393,0.195094,0.343052,0.533501
4,1,4,いつもと違って、受け答えがおかしいとか、様子が変だということはありますか。 はい、あります。...,0.266853,0.180636,0.381274,0.439162
5,2,0,救急ですか、火事ですか。 救急です。職場の同僚がひどい頭痛で倒れ込みました。,0.336934,0.142117,0.437919,0.558986
6,2,1,どうしましたか。 休憩中に急に頭が割れるみたいに痛いと言って、その場にしゃがみ込んでしまいました。,0.293870,0.125630,0.344964,0.513604
7,2,2,痛みはじわじわではなく、急に始まった感じですか。 はい、急でした。普通に話していたのに、突然です。,0.251174,0.113967,0.439027,0.406220
8,2,3,しびれたり、顔や腕が動かしにくそうな様子はありますか。 左腕がしびれると言っています。コップ...,0.349039,0.125966,0.273418,0.518907
9,2,4,普段と違う行動や、会話がかみ合わない感じはありますか。 あります。名前を呼んでも反応が鈍いし...,0.341741,0.201229,0.458283,0.552229


## 4.2.cos類似度を元に入力するペアの選定

In [221]:
########### 閾値の設定 ###########
threshold=0.02
##################################

In [222]:
def select_pairs_by_similarity_gap(group_df, cos_sim_col_name, threshold=0.1):
    # グループ内でコサイン類似度の降順にソートし、元のインデックスを保持したままリセット
    sorted_group_indexed = group_df.sort_values(by=cos_sim_col_name, ascending=False)
    sorted_group_temp = sorted_group_indexed.reset_index(drop=True)

    # 初期値としてFalseを設定: デフォルトでは全て不採用
    adopted_temp = pd.Series(False, index=range(len(sorted_group_temp)))

    # ペアが存在する場合、最も類似度が高いペアは常に採用する
    if len(sorted_group_temp) > 0:
        adopted_temp.iloc[0] = True

    # ペアが1つ以下の場合、ギャップの判断はできないため、ここまでで確定した採用状況を返す
    if len(sorted_group_temp) <= 1:
        # 元のインデックスに再構築して返す
        return adopted_temp.set_axis(sorted_group_indexed.index).reindex(group_df.index)

    # 連続するコサイン類似度間の差を計算 (前の値 - 次の値)
    # diffs.iloc[i] は sorted_group_temp.iloc[i-1][cos_sim_col_name] - sorted_group_temp.iloc[i][cos_sim_col_name] に相当
    diffs = sorted_group_temp[cos_sim_col_name].diff() * -1

    # 2番目の要素からループを開始し、ギャップロジックを適用
    # 最初の要素は既に採用済み
    for i in range(1, len(diffs)):
        current_gap = diffs.iloc[i]
        if current_gap > threshold:
            # ギャップが閾値を超えた場合、その時点から後続のペアは全て不採用とする
            adopted_temp.iloc[i:] = False
            break
        else:
            # ギャップが閾値以下の場合は、このペアを採用する
            adopted_temp.iloc[i] = True

    # 一時的なTrue/Falseシリーズを元のDataFrameのインデックスにマッピングし直す
    # sorted_group_indexed.indexを使って元のソートされたインデックスをセットし、
    # その後group_df.indexにリインデックスして、元の行順に戻す
    final_adopted_series = adopted_temp.set_axis(sorted_group_indexed.index).reindex(group_df.index)
    return final_adopted_series

In [223]:
# 各'ひし形'に対して選定ロジックを適用

# hisigata DataFrameの列名を直接使用
for col_name in hisigata.columns:
    cos_sim_col = f'cos_sim_{col_name}'
    adopted_col = f'採用ペア_{col_name}'

    # 'id'でグループ化し、選定関数を適用
    # group_keys=False は、グループ化キーを結果のインデックスに含めないようにするため
    df_pairs = df_pairs.groupby('id', group_keys=False).apply(
        lambda group: group.assign(**{adopted_col: select_pairs_by_similarity_gap(group, cos_sim_col, threshold)})
    )

# 結果の表示（関連カラムのみ）
print('閾値:', threshold)
# 表示カラムも動的に生成
display_cols = ['id', 'ペア番号', 'ペア', '質問', '回答']
for col_name in hisigata.columns:
    display_cols.append(f'cos_sim_{col_name}')
    display_cols.append(f'採用ペア_{col_name}')
display(df_pairs[display_cols])

閾値: 0.02


/tmp/ipykernel_14496/465232545.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_pairs = df_pairs.groupby('id', group_keys=False).apply(
/tmp/ipykernel_14496/465232545.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_pairs = df_pairs.groupby('id', group_keys=False).apply(
/tmp/ipykernel_14496/465232545.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This beha

,id,ペア番号,ペア,質問,回答,cos_sim_ひし形_全通り_頭痛_1,採用ペア_ひし形_全通り_頭痛_1,cos_sim_ひし形_全通り_頭痛_2,採用ペア_ひし形_全通り_頭痛_2,cos_sim_ひし形_全通り_頭痛_3,採用ペア_ひし形_全通り_頭痛_3,cos_sim_ひし形_直接,採用ペア_ひし形_直接
0,1,0,救急ですか、火事ですか。 救急です。夫の様子がおかしいです。,救急ですか、火事ですか。,救急です。夫の様子がおかしいです。,0.333049,False,0.114088,False,0.417469,False,0.579390,True
1,1,1,どうしましたか。 さっきテレビを見ていたら、急に頭を押さえて、ものすごく痛いって言い出しまし...,どうしましたか。,さっきテレビを見ていたら、急に頭を押さえて、ものすごく痛いって言い出しました。頭痛です。,0.371689,True,0.142192,False,0.481872,False,0.500633,False
2,1,2,その頭の痛みは、急に強く出たんですね。 はい、突然です。今までにないくらい痛いと言っています。,その頭の痛みは、急に強く出たんですね。,はい、突然です。今までにないくらい痛いと言っています。,0.343067,False,0.164993,True,0.551439,True,0.481516,False
3,1,3,手足のしびれや、片側が動かしにくい感じはありますか。 あります。右手がうまく動かないみたいで...,手足のしびれや、片側が動かしにくい感じはありますか。,あります。右手がうまく動かないみたいで、しびれるとも言っています。,0.372393,True,0.195094,True,0.343052,False,0.533501,False
4,1,4,いつもと違って、受け答えがおかしいとか、様子が変だということはありますか。 はい、あります。...,いつもと違って、受け答えがおかしいとか、様子が変だということはありますか。,はい、あります。呼びかけても返事が遅いですし、言葉も少し変で、ぼんやりしています。,0.266853,False,0.180636,True,0.381274,False,0.439162,False
5,2,0,救急ですか、火事ですか。 救急です。職場の同僚がひどい頭痛で倒れ込みました。,救急ですか、火事ですか。,救急です。職場の同僚がひどい頭痛で倒れ込みました。,0.336934,True,0.142117,False,0.437919,True,0.558986,True
6,2,1,どうしましたか。 休憩中に急に頭が割れるみたいに痛いと言って、その場にしゃがみ込んでしまいました。,どうしましたか。,休憩中に急に頭が割れるみたいに痛いと言って、その場にしゃがみ込んでしまいました。,0.293870,False,0.125630,False,0.344964,False,0.513604,False
7,2,2,痛みはじわじわではなく、急に始まった感じですか。 はい、急でした。普通に話していたのに、突然です。,痛みはじわじわではなく、急に始まった感じですか。,はい、急でした。普通に話していたのに、突然です。,0.251174,False,0.113967,False,0.439027,True,0.406220,False
8,2,3,しびれたり、顔や腕が動かしにくそうな様子はありますか。 左腕がしびれると言っています。コップ...,しびれたり、顔や腕が動かしにくそうな様子はありますか。,左腕がしびれると言っています。コップも落としてしまいました。,0.349039,True,0.125966,False,0.273418,False,0.518907,False
9,2,4,普段と違う行動や、会話がかみ合わない感じはありますか。 あります。名前を呼んでも反応が鈍いし...,普段と違う行動や、会話がかみ合わない感じはありますか。,あります。名前を呼んでも反応が鈍いし、何を聞いても同じことを繰り返しています。,0.341741,True,0.201229,True,0.458283,True,0.552229,True


# 5.csvにエクスポート

In [224]:
# # Google Driveの適切なパスにdf_pairsをCSVとしてエクスポート
# output_path = '/content/drive/MyDrive/NTCIR-19/input_pairs.csv'
# df_pairs.to_csv(output_path, index=False, encoding='utf-8')

# print(f"df_pairs を {output_path} にエクスポートしました。")

df_pairs を /content/drive/MyDrive/NTCIR-19/input_pairs.csv にエクスポートしました。
